## Running kinGEMs Pipeline on $E. coli$ iML1515 Model

In [14]:

%load_ext autoreload
%autoreload 2

import os
import sys
import pandas as pd # type: ignore
import matplotlib.pyplot as plt # type: ignore
import cobra # type: ignore
import numpy as np # type: ignore
from datetime import datetime
import random
from cobra.io import write_sbml_model # type: ignore

# Add parent directory to Python path
sys.path.append(os.path.abspath('..'))

# Import kinGEMs components
import kinGEMs
from kinGEMs.dataset import load_model, convert_to_irreversible, prepare_model_data, retrieve_sequences, map_metabolites, merge_substrate_sequences, process_merged_data_with_folds, process_kcat_predictions, assign_kcats_to_model, format_kcats_like_gpr, annotate_model_with_kcat_and_gpr
from kinGEMs.modeling.optimize import run_optimization_with_dataframe
from kinGEMs.modeling.tuning import simulated_annealing, analyze_kcat_changes
from kinGEMs.modeling.fva import flux_variability_analysis, plot_flux_variability, plot_cumulative_fvi_distribution


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Set your GEM name here

In [5]:

# === Configuration ===
organism_strain_GEMname = "ecoli_iML1515" # Update this
organism = "E coli" # Update this 
run_id = f"{organism_strain_GEMname}_{datetime.today().strftime('%Y%m%d')}_{random.randint(1000, 9999)}"

# Directories
data_dir = "../data"
raw_data_dir = os.path.join(data_dir, "raw")
interim_data_dir = os.path.join(data_dir, "interim")
interim_data_dir_spec = os.path.join(interim_data_dir, f"{organism_strain_GEMname}")
processed_data_dir = os.path.join(data_dir, "processed")
processed_data_dir_spec = os.path.join(processed_data_dir, f"{organism_strain_GEMname}")
CPIPred_data_dir = os.path.join(interim_data_dir, "CPI-Pred predictions")
results_dir = os.path.join(os.getcwd(), "results")
tuning_results_dir = os.path.join(results_dir, "tuning_results", run_id)
os.makedirs(tuning_results_dir, exist_ok=True)

# Input files
model_path = os.path.join(raw_data_dir, "iML1515_GEM.xml") # Update this
predictions_csv_path = os.path.join(CPIPred_data_dir, f"X06A_kinGEMs_{organism_strain_GEMname}_predictions.csv")

# Output files
substrates_output = os.path.join(interim_data_dir_spec, f"{organism_strain_GEMname}_substrates.csv")
sequences_output = os.path.join(interim_data_dir_spec, f"{organism_strain_GEMname}_sequences.csv")
merged_data_output = os.path.join(interim_data_dir_spec, f"{organism_strain_GEMname}_merged_data.csv")
processed_data_output = os.path.join(processed_data_dir_spec, f"{organism_strain_GEMname}_processed_data.csv")

# Simulation parameters
biomass_reaction = 'BIOMASS_Ec_iML1515_WT_75p37M' # Update this
enzyme_upper_bound = 0.125


### Step 1: Preparing and processing model data

In [7]:

print("=== Step 1: Preparing model data ===")
irrev_model, substrate_df, sequences_df = prepare_model_data(
    model_path=model_path,
    substrates_output=substrates_output,
    sequences_output=sequences_output,
    organism=organism
)


=== Step 1: Preparing model data ===
Loaded model with 2712 reactions and 1877 metabolites
Read LP format model from file C:\Users\Rana\AppData\Local\Temp\tmp_wj2o6ql.lp


2025-05-23 18:34:53,249 - gurobipy - INFO - Read LP format model from file C:\Users\Rana\AppData\Local\Temp\tmp_wj2o6ql.lp


Reading time = 0.02 seconds


2025-05-23 18:34:53,249 - gurobipy - INFO - Reading time = 0.02 seconds


: 1877 rows, 5424 columns, 21150 nonzeros


...

Number of reactions being added from exchange: 970
Converted to irreversible model with 3682 reactions
Extracted 6194 substrate-reaction pairs


c:\Users\Rana\OneDrive\Documents\GitHub\kinGEMs_v2\kinGEMs\dataset.py:215: DtypeWarning: Columns (4,10) have mixed types. Specify dtype option on import or set low_memory=False.
  SEED_comps = pd.read_csv(SEED_COMPOUNDS, sep='\t')
2025-05-23 18:35:34,484 - kinGEMs.dataset - INFO - There are 1814 substrates in the GEM.
2025-05-23 18:35:34,516 - kinGEMs.dataset - INFO - -----------------------------
2025-05-23 18:35:34,516 - kinGEMs.dataset - INFO - Mapping substrate: ala__D_c
2025-05-23 18:35:34,547 - kinGEMs.dataset - INFO - BiGG Name: D-Alanine
2025-05-23 18:35:34,591 - kinGEMs.dataset - INFO - SMILES found in SEED: C[C@@H]([NH3+])C(=O)[O-]
2025-05-23 18:35:34,592 - kinGEMs.dataset - INFO - -----------------------------
2025-05-23 18:35:34,592 - kinGEMs.dataset - INFO - Mapping substrate: pydx5p_c
2025-05-23 18:35:34,620 - kinGEMs.dataset - INFO - BiGG Name: Pyridoxal 5'-phosphate
2025-05-23 18:35:34,659 - kinGEMs.dataset - INFO - SMILES found in MetaNetX: Cc1ncc(COP(=O)([O-])[O-])c(C

Mapped metabolites to SMILES (5685 found)


2025-05-23 20:22:15,721 - root - WARNING - No sequence found for gene s0001


Retrieved 1515 protein sequences


### Step 2: Merging substrate and sequence data

In [8]:

print("=== Step 2: Merging substrate and sequence data ===")
merged_data = merge_substrate_sequences(
    substrate_df=substrate_df,
    sequences_df=sequences_df,
    model=irrev_model,
    output_path=merged_data_output
)


=== Step 2: Merging substrate and sequence data ===


### Step 3: Processing CPI-Pred kcat values

In [9]:

print("=== Step 3: Processing CPI-Pred kcat values & annotating model ===")
processed_data = process_kcat_predictions(
    merged_df=merged_data,
    predictions_csv_path=predictions_csv_path,
    output_path=processed_data_output
)

irrev_model = annotate_model_with_kcat_and_gpr(
    model=irrev_model,
    df=processed_data
)

(solution_value, df_FBA, gene_sequences_dict, _)=run_optimization_with_dataframe(
    model=irrev_model, 
    processed_df=processed_data, 
    objective_reaction=biomass_reaction, 
    enzyme_upper_bound=enzyme_upper_bound, 
    enzyme_ratio=True, 
    maximization=True, 
    multi_enzyme_off=False, 
    isoenzymes_off=False, 
    promiscuous_off=False, 
    complexes_off=False,
    output_dir=None, save_results=False)

print("Biomass value: ", solution_value)


=== Step 3: Processing CPI-Pred kcat values & annotating model ===
Optimization completed successfully!
Total reaction-gene pairs: 6253
Enzyme-constrained pairs: 5946
Unconstrained pairs (missing data): 307
Promiscuous enzymes in system: 1488
Isoenzymatic passes: 22217
Enzyme complex reactions: 1914
Biomass value:  0.0001032462986966647


### Step 4: Simulated Annealing

In [ ]:

print("=== Step 4: Running simulated annealing ===")
temperature = 1.0
cooling_rate = 0.98
min_temperature = 0.01
max_iterations = 100
max_unchanged_iterations = 3
change_threshold = 0.001
biomass_goal = 0.2

kcat_dict, top_targets, df_new, iterations, biomasses, df_FBA = simulated_annealing(
    model=irrev_model,
    processed_data=processed_data,
    biomass_reaction=biomass_reaction,
    objective_value=biomass_goal,
    gene_sequences_dict=gene_sequences_dict,      # ← new
    output_dir=tuning_results_dir,
    enzyme_fraction=enzyme_upper_bound,
    temperature=temperature,
    cooling_rate=cooling_rate,
    min_temperature=min_temperature,
    max_iterations=max_iterations,
    max_unchanged_iterations=max_unchanged_iterations,
    change_threshold=change_threshold
)

print(f"Final biomass: {biomasses[-1]:.4f}")
print(f"Improvement: {(biomasses[-1] - biomasses[0]) / biomasses[0] * 100:.1f}%")
print("Top‐25 enzymes by mass contribution:")
print(top_targets[['Reactions','Single_gene','enzyme_mass']])


=== Step 4: Running simulated annealing ===
Optimization completed successfully!
Total reaction-gene pairs: 6253
Enzyme-constrained pairs: 5946
Unconstrained pairs (missing data): 307
Promiscuous enzymes in system: 1488
Isoenzymatic passes: 22217
Enzyme complex reactions: 1914
Optimization completed successfully!
Total reaction-gene pairs: 6253
Enzyme-constrained pairs: 5946
Unconstrained pairs (missing data): 307
Promiscuous enzymes in system: 1488
Isoenzymatic passes: 22217
Enzyme complex reactions: 1914
Optimization completed successfully!
Total reaction-gene pairs: 6253
Enzyme-constrained pairs: 5946
Unconstrained pairs (missing data): 307
Promiscuous enzymes in system: 1488
Isoenzymatic passes: 22217
Enzyme complex reactions: 1914
Optimization completed successfully!
Total reaction-gene pairs: 6253
Enzyme-constrained pairs: 5946
Unconstrained pairs (missing data): 307
Promiscuous enzymes in system: 1488
Isoenzymatic passes: 22217
Enzyme complex reactions: 1914


### Step 5: FVA 

In [ ]:

print("=== Step 5: Running Flux Variability Analysis ===")
fva_results_path = os.path.join(tuning_results_dir, f"{organism_strain_GEMname}_fva_results.csv")
fva_plot_path = os.path.join(tuning_results_dir, f"{organism_strain_GEMname}_fva_flux_range_plot.png")
fva_cumulative_path = os.path.join(tuning_results_dir, f"{organism_strain_GEMname}_fva_cumulative_plot.png")

fva_results, _, _ = flux_variability_analysis(
    model=irrev_model,
    processed_df=df_new,
    biomass_reaction=biomass_reaction,
    output_file=fva_results_path,
    enzyme_upper_bound=enzyme_upper_bound
)

# Plot standard FVA range
fig = plot_flux_variability(fva_results, output_file=fva_plot_path)

# Plot cumulative distribution
plot_cumulative_fvi_distribution(
    dfs=[fva_results],
    labels=[organism_strain_GEMname],
    output_file=fva_cumulative_path
)


### Step 6: Save Final Model

In [ ]:
# Define output path for final GEM
model_output_dir = os.path.join("models")
os.makedirs(model_output_dir, exist_ok=True)
model_output_path = os.path.join(model_output_dir, f"{run_id}.xml")

# After simulated annealing
model_with_kcats = assign_kcats_to_model(irrev_model, df_new)

# Preview:
format_kcats_like_gpr(model_with_kcats.reactions.get_by_id("PGI"))


# Save the final irreversible model
write_sbml_model(model_with_kcats, model_output_path)

print(f"Final GEM saved to: {model_output_path}")